# Clase 6. Control sin modelo: SARSA y Q-learning

**Duración estimada:** 105 minutos
**Capítulo del manual:** [07_control_sin_modelo.md](../manual/07_control_sin_modelo.md)

## De evaluar a mejorar

En la clase anterior aprendiste a evaluar una política fija: dado un agente que siempre actúa igual, calculaste cuánto espera ganar desde cada estado, con Monte Carlo (MC) y con Diferencia Temporal (TD, de *Temporal Difference*). Eso es **predicción**, y deja una pregunta abierta: ¿y si la política de partida no es buena? Medir el valor de una estrategia mediocre no la convierte en una estrategia mejor.

Hoy das el siguiente paso: el **control**. Ya no te conformas con medir cuánto vale una política, quieres que el agente la mejore mientras aprende, episodio tras episodio. Vas a conocer dos algoritmos centrales del aprendizaje por refuerzo (RL, de *Reinforcement Learning*) moderno: SARSA (de *State-Action-Reward-State-Action*) y Q-learning. Ambos aprenden sin modelo —sin conocer de antemano cómo funciona el entorno— y ambos parten de la misma pregunta ("¿qué tan buena es esta acción en este estado?"), pero responden de forma distinta, con consecuencias prácticas muy diferentes.

### Objetivos de hoy

- Explicar la Iteración de Política Generalizada (GPI) y por qué el control necesita Q(s,a) en vez de V(s)
- Distinguir la actualización on-policy de SARSA de la actualización off-policy de Q-learning
- Implementar el entorno Cliff Walking y entrenar ambos algoritmos sobre él
- Comparar la recompensa durante el entrenamiento y las rutas finales aprendidas por cada uno

> 🎙️ **Guion de clase:** abre preguntando "si ya sabéis medir cuánto vale una estrategia, ¿cómo la mejoraríais?". Deja un par de respuestas antes de introducir GPI: la idea de alternar evaluar y mejorar suele salir sola.


## 1. De predecir a mejorar: Iteración de Política Generalizada

Imagina que aprendes a cocinar un plato nuevo. Lo preparas, lo pruebas, ajustas la receta un poco, vuelves a prepararlo, lo pruebas otra vez y ajustas de nuevo. No esperas a dominar la receta perfecta en la teoría antes de cocinar por primera vez: alternas entre "evaluar cómo quedó" y "mejorar la receta", una y otra vez, hasta acercarte a algo excelente.

Esa alternancia es la idea detrás de la **Iteración de Política Generalizada** (GPI, de *Generalized Policy Iteration*), un principio que estructura casi todo el control en RL:

1. **Evaluación**: estima cuánto vale cada acción bajo la política actual
2. **Mejora**: ajusta la política para que favorezca las acciones con mejor valor estimado
3. **Repite**: vuelve al paso uno con la política mejorada

No necesitas completar cada paso a la perfección para que funcione: puedes evaluar un poco, mejorar un poco, evaluar un poco más, y esa versión "a medias" converge igualmente hacia la política óptima en el caso tabular (cuando guardas un valor explícito por cada combinación de estado y acción). SARSA y Q-learning son dos formas distintas de ejecutar GPI paso a paso, actualizando el valor después de cada acción individual en vez de esperar a que termine el episodio completo.

Una diferencia importante frente a la clase anterior: en control ya no basta con estimar V(s). Como no tienes un modelo del entorno, necesitas el valor de cada **acción** en cada estado, Q(s,a): así puedes mejorar la política eligiendo la acción con mayor Q, sin simular qué pasaría con cada alternativa.


## 2. El dilema de explorar: ε-greedy y GLIE

Si el agente siempre elige la acción que hoy parece mejor, nunca descubre si otra acción, todavía no probada, es en realidad superior. Por eso el control necesita explorar. La estrategia más habitual es la política **ε-greedy** (epsilon-greedy): con probabilidad ε el agente elige una acción al azar (explora), y con probabilidad 1−ε elige la acción con mayor valor Q conocido hasta ahora (explota).

Para garantizar que este proceso converja de verdad hacia la política óptima existe una condición formal llamada **GLIE** (de *Greedy in the Limit with Infinite Exploration*, "codiciosa en el límite con exploración infinita"). Exige dos cosas: que cada combinación de estado y acción se visite un número infinito de veces, y que la política se vuelva completamente codiciosa con el tiempo. Decaer ε de forma progresiva es la manera habitual de cumplirla.

También existe el **control por Monte Carlo**: evalúas Q(s,a) promediando retornos completos de episodios y mejoras la política haciéndola ε-greedy respecto a esos valores. Funciona, pero necesita episodios completos y tiende a tener más varianza. SARSA y Q-learning resuelven el mismo problema actualizando en cada paso, gracias al bootstrapping de TD.

> 🎙️ **Guion de clase:** conecta con la clase de bandidos: ε-greedy ya apareció allí. Pregunta "¿por qué aquí también hace falta explorar, si ya conocemos el entorno un poco mejor que en un bandido?".


## 3. SARSA: aprende la política que realmente sigues

SARSA toma su nombre de la secuencia que usa en cada actualización: estado, acción, recompensa, estado siguiente, acción siguiente. El agente está en un estado, elige una acción con su política ε-greedy, observa la recompensa y el nuevo estado, y entonces —el paso clave— **elige también la siguiente acción con esa misma política ε-greedy**, antes de actualizar. La actualización usa el Q de esa acción siguiente, la que realmente se va a ejecutar:

```
Q(S,A) se acerca un poco a: recompensa recibida + valor descontado de Q en el estado siguiente,
                             para la acción que la política ε-greedy eligió de verdad
```

Por eso SARSA es **on-policy** (dentro de política): aprende el valor de la política que efectivamente sigue, incluida su propia aleatoriedad exploratoria. Si tu política ε-greedy a veces comete "errores" al explorar, SARSA incorpora esos errores en su estimación de valor, y eso lo hace más prudente: aprende a evitar zonas donde una exploración desafortunada podría salir cara.


## 4. Q-learning: aprende la política óptima, aunque explores de otra forma

Q-learning cambia un solo detalle, con consecuencias grandes. En vez de usar el Q de la acción que realmente vas a tomar a continuación, usa el valor **máximo** posible entre todas las acciones del estado siguiente, sin importar cuál elijas después de verdad:

```
Q(S,A) se acerca un poco a: recompensa recibida + valor descontado del mejor Q posible
                             en el estado siguiente (el máximo sobre todas las acciones)
```

Esto convierte a Q-learning en **off-policy** (fuera de política): la política de comportamiento (con la que exploras, por ejemplo ε-greedy) puede ser distinta de la política objetivo que aprendes (la puramente codiciosa, la óptima). Q-learning aprende el valor de "si a partir de ahora actuara siempre de forma óptima", incluso mientras sigue explorando de forma imperfecta. Esta separación entre "cómo exploro" y "qué aprendo" también permite aprender a partir de datos recogidos con otra política, o incluso ya registrados de antemano.

> 🎙️ **Guion de clase:** escribe las dos fórmulas una encima de la otra en la pizarra y rodea la única diferencia: `Q(S',A')` (la acción que de verdad tomarás) frente a `max Q(S',·)` (la mejor acción posible). Es todo lo que distingue a los dos algoritmos.


## 5. La analogía del acantilado: prudente frente a arriesgado

La forma más clara de sentir la diferencia es el ejemplo clásico del **acantilado** (*Cliff Walking*). Una cuadrícula donde el agente empieza en la esquina inferior izquierda y el objetivo está en la esquina inferior derecha. Entre ambos, a lo largo del borde inferior, hay un acantilado: caer en él da una penalización fuerte y te devuelve al punto de partida. Cada paso normal cuesta un poco, así que conviene llegar cuanto antes.

El camino más corto bordea el acantilado, pegado al borde. Es también el más arriesgado, porque un solo paso en falso —un movimiento exploratorio aleatorio de ε-greedy— te tira por el borde.

- **Q-learning** aprende que ese camino corto y pegado al borde es el óptimo, porque calcula el valor asumiendo que a partir de ahora actuará siempre de forma óptima, sin considerar que su propia exploración podría hacerle caer. La ruta final, una vez dejas de explorar, es la más corta posible
- **SARSA** aprende una ruta más alejada del borde, más larga pero más segura, porque su estimación sí tiene en cuenta que, mientras explora, a veces se equivoca

Durante el propio entrenamiento, mientras ambos agentes siguen explorando con ε-greedy, SARSA suele acumular más recompensa por episodio porque cae menos veces por el acantilado. Q-learning cae más durante el entrenamiento, pero termina aprendiendo la política objetivo más corta y óptima. Ninguno "gana" en abstracto: cada uno responde a una pregunta distinta sobre qué política estás evaluando.

> 🎙️ **Guion de clase:** antes de programar, pide a la clase que prediga en voz alta cuál de las dos rutas dibujaría cada algoritmo. Guarda la predicción para contrastarla con el gráfico de rutas al final.


## 6. Manos a la obra: Cliff Walking con SARSA y Q-learning

Vas a construir el entorno del acantilado desde cero, sin librerías externas, y a entrenar un agente con cada algoritmo. Solo necesitas `numpy` y `matplotlib`, que ya vienen preinstalados en Google Colab.

Construye el código célula a célula, ejecutando cada una antes de pasar a la siguiente.


In [ ]:
# Solo necesitas numpy y matplotlib, ya vienen preinstalados en Google Colab
import numpy as np
import matplotlib.pyplot as plt


### El entorno: la cuadrícula y el acantilado

Una cuadrícula de cuatro filas por 12 columnas (4×12). El agente empieza abajo a la izquierda (fila 3, columna 0) y el objetivo está abajo a la derecha (fila 3, columna 11). Toda la fila inferior entre ambos, excepto el inicio y el objetivo, es el acantilado.

Las acciones son cuatro: 0=arriba, 1=abajo, 2=izquierda, 3=derecha.


In [ ]:
FILAS, COLUMNAS = 4, 12
INICIO = (3, 0)
META = (3, 11)
ACANTILADO = [(3, c) for c in range(1, 11)]
NUM_ACCIONES = 4


### La función `paso`: transición y recompensa

Dado un estado y una acción, `paso` devuelve el estado siguiente, la recompensa recibida y si el episodio terminó. Caer por el acantilado penaliza fuerte (−100) y devuelve al agente al inicio, sin terminar el episodio. Llegar a la meta da −1 y termina. Cualquier otro movimiento cuesta −1, el precio de cada paso.

> 🎙️ **Guion de clase:** pregunta "¿por qué caer por el acantilado no termina el episodio?". Respuesta: así el agente sigue aprendiendo dentro del mismo episodio en vez de reiniciar el conteo de recompensa.


In [ ]:
def paso(estado, accion):
    """Aplica una acción y devuelve (nuevo_estado, recompensa, terminado)."""
    fila, col = estado
    if accion == 0:
        fila = max(fila - 1, 0)
    elif accion == 1:
        fila = min(fila + 1, FILAS - 1)
    elif accion == 2:
        col = max(col - 1, 0)
    elif accion == 3:
        col = min(col + 1, COLUMNAS - 1)

    nuevo_estado = (fila, col)

    if nuevo_estado in ACANTILADO:
        # Caer por el acantilado: penalización fuerte y vuelta al inicio
        return INICIO, -100, False
    if nuevo_estado == META:
        return nuevo_estado, -1, True
    return nuevo_estado, -1, False


### Comprobación rápida

Antes de seguir, verifica que `paso` se comporta como esperas.

> 🎙️ **Guion de clase:** pide a alguien que prediga en voz alta el resultado de `paso((2, 1), 1)` (moverse "abajo" hacia el acantilado) antes de ejecutar la celda.


In [ ]:
print("Moverse 'abajo' desde (2, 1), justo encima del acantilado:", paso((2, 1), 1))
print("Moverse 'derecha' desde el inicio (3, 0):", paso(INICIO, 3))
print("Moverse 'arriba' desde el inicio (3, 0):", paso(INICIO, 0))


### La política ε-greedy

Con probabilidad `epsilon` elige una acción al azar (explora); si no, elige la acción con mayor Q conocido (explota). Si hay empate entre varias acciones, `np.random.choice` desempata al azar en vez de quedarse siempre con la primera.


In [ ]:
def politica_epsilon_greedy(Q, estado, epsilon):
    """Con probabilidad epsilon explora; si no, elige la mejor acción conocida."""
    if np.random.rand() < epsilon:
        return np.random.randint(NUM_ACCIONES)
    valores = Q[estado]
    return np.random.choice(np.flatnonzero(valores == valores.max()))


### Entrenar con SARSA

Fíjate en la línea clave: `siguiente_accion` se elige **antes** de actualizar, con la misma política ε-greedy, y es ese Q el que entra en el objetivo. Eso es lo que hace a SARSA on-policy.


In [ ]:
def entrenar_sarsa(num_episodios=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    Q = np.zeros((FILAS, COLUMNAS, NUM_ACCIONES))
    recompensas_por_episodio = []

    for _ in range(num_episodios):
        estado = INICIO
        accion = politica_epsilon_greedy(Q, estado, epsilon)
        recompensa_total = 0
        terminado = False

        while not terminado:
            siguiente_estado, recompensa, terminado = paso(estado, accion)
            siguiente_accion = politica_epsilon_greedy(Q, siguiente_estado, epsilon)

            # Actualización SARSA: usa el Q de la acción que SÍ se va a tomar
            objetivo = recompensa + gamma * Q[siguiente_estado][siguiente_accion]
            Q[estado][accion] += alpha * (objetivo - Q[estado][accion])

            estado, accion = siguiente_estado, siguiente_accion
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    return Q, recompensas_por_episodio


### Entrenar con Q-learning

Compara esta función con la anterior: la estructura del bucle es casi idéntica, pero la actualización ya no depende de qué acción elijas después, sino del **máximo** Q del estado siguiente. Esa es la única diferencia real entre los dos algoritmos.


In [ ]:
def entrenar_q_learning(num_episodios=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    Q = np.zeros((FILAS, COLUMNAS, NUM_ACCIONES))
    recompensas_por_episodio = []

    for _ in range(num_episodios):
        estado = INICIO
        recompensa_total = 0
        terminado = False

        while not terminado:
            accion = politica_epsilon_greedy(Q, estado, epsilon)
            siguiente_estado, recompensa, terminado = paso(estado, accion)

            # Actualización Q-learning: usa el MÁXIMO Q del estado siguiente,
            # sin importar qué acción se elija después en la práctica
            mejor_valor_siguiente = Q[siguiente_estado].max()
            objetivo = recompensa + gamma * mejor_valor_siguiente
            Q[estado][accion] += alpha * (objetivo - Q[estado][accion])

            estado = siguiente_estado
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    return Q, recompensas_por_episodio


### Extraer la ruta aprendida

Una vez entrenado, `extraer_ruta_greedy` sigue la política puramente codiciosa (sin exploración, `epsilon=0`) desde el inicio. Es la ruta que el agente elegiría si dejaras de explorar del todo: la política que de verdad aprendió.


In [ ]:
def extraer_ruta_greedy(Q, max_pasos=50):
    """Sigue la política puramente codiciosa (sin exploración) desde el inicio."""
    estado = INICIO
    ruta = [estado]
    for _ in range(max_pasos):
        if estado == META:
            break
        accion = np.argmax(Q[estado])
        estado, _, terminado = paso(estado, accion)
        ruta.append(estado)
        if terminado:
            break
    return ruta


### Ejecuta el entrenamiento

Fija la semilla aleatoria para que el resultado sea reproducible y entrena los dos agentes con los mismos hiperparámetros: 500 episodios, α=0.5, γ=1.0, ε=0.1.

> 🎙️ **Guion de clase:** mientras se ejecuta (es casi instantáneo), recuerda en voz alta la frase clave de hoy: "SARSA aprende el valor de lo que hace de verdad; Q-learning aprende el valor de lo óptimo".


In [ ]:
np.random.seed(0)
Q_sarsa, recompensas_sarsa = entrenar_sarsa()
Q_qlearning, recompensas_qlearning = entrenar_q_learning()

print("Episodios entrenados:", len(recompensas_sarsa))


### Gráfico 1: recompensa por episodio durante el entrenamiento

Se suaviza con una media móvil porque episodio a episodio hay mucho ruido: una sola caída por el acantilado (−100) desestabiliza el número bruto.


In [ ]:
def media_movil(valores, ventana=20):
    return np.convolve(valores, np.ones(ventana) / ventana, mode="valid")

plt.figure(figsize=(9, 5))
plt.plot(media_movil(recompensas_sarsa), label="SARSA (on-policy)")
plt.plot(media_movil(recompensas_qlearning), label="Q-learning (off-policy)")
plt.xlabel("Episodio")
plt.ylabel("Recompensa por episodio (media móvil)")
plt.title("Cliff Walking: recompensa durante el entrenamiento")
plt.legend()
plt.tight_layout()
plt.show()


**Cómo leer el gráfico:** la curva de SARSA suele estar por encima de la de Q-learning durante buena parte del entrenamiento, porque cae menos veces por el acantilado mientras explora. Q-learning paga, durante el entrenamiento, el precio de calcular su objetivo asumiendo una política óptima que en la práctica todavía no sigue.

> 🎙️ **Guion de clase:** pregunta "¿por qué la línea de Q-learning es más baja aquí, si dijimos que aprende la ruta óptima?". Deja que la clase conecte la respuesta con la diferencia on-policy / off-policy antes de continuar.


### Gráfico 2: rutas finales aprendidas (política greedy)

Ahora extrae la ruta que cada agente seguiría sin explorar, y dibújala sobre la cuadrícula.


In [ ]:
def dibujar_ruta(ax, ruta, titulo):
    cuadricula = np.zeros((FILAS, COLUMNAS))
    for (f, c) in ACANTILADO:
        cuadricula[f, c] = -1
    ax.imshow(cuadricula, cmap="Pastel1")
    filas_ruta = [p[0] for p in ruta]
    columnas_ruta = [p[1] for p in ruta]
    ax.plot(columnas_ruta, filas_ruta, marker="o", color="tab:blue")
    ax.scatter(*INICIO[::-1], color="green", s=100, label="Inicio", zorder=5)
    ax.scatter(*META[::-1], color="red", s=100, label="Meta", zorder=5)
    ax.set_title(titulo)
    ax.invert_yaxis()
    ax.legend(loc="upper center")

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
dibujar_ruta(ejes[0], extraer_ruta_greedy(Q_sarsa), "Ruta final de SARSA (más prudente)")
dibujar_ruta(ejes[1], extraer_ruta_greedy(Q_qlearning), "Ruta final de Q-learning (más corta)")
plt.tight_layout()
plt.show()


**Cómo leer el gráfico:** la ruta de Q-learning suele pegarse al borde del acantilado (fila justo encima de él): es la más corta y la óptima. La ruta de SARSA se aleja una o más filas hacia arriba, sacrificando longitud a cambio de seguridad frente a su propia exploración. Contrasta esto con la predicción que hizo la clase antes de programar.

> 🎙️ **Guion de clase:** pide a la clase que cuente cuántos pasos tiene cada ruta y que relacione la diferencia con la recompensa del gráfico anterior: más pasos, más veces se paga el coste de −1 por movimiento.


## 7. Sesgo de sobreestimación y Double Q-learning

Q-learning tiene un efecto secundario sutil: como su actualización usa el **máximo** entre varias estimaciones ruidosas de Q, ese máximo tiende a estar sesgado hacia arriba, igual que si eliges siempre "el mejor candidato" entre varias mediciones imprecisas, sueles quedarte con una que exagera por azar. Esto se conoce como **sesgo de sobreestimación**.

**Double Q-learning** lo corrige manteniendo dos tablas de valores independientes: una decide qué acción parece mejor, y la otra evalúa cuánto vale realmente esa acción (alternando los papeles en cada paso). Al separar "quién elige" de "quién evalúa", se elimina buena parte del sesgo.

También existe **SARSA esperado**: en vez de usar el Q de la acción que realmente se toma (SARSA) o el máximo (Q-learning), usa el promedio ponderado de Q sobre todas las acciones posibles, según la probabilidad de cada una bajo la política. Cuando la política objetivo es completamente codiciosa, esta versión coincide exactamente con Q-learning.


## Puntos clave

- El control busca mejorar la política, no solo evaluarla: alterna evaluación y mejora siguiendo el principio de Iteración de Política Generalizada (GPI)
- La política ε-greedy equilibra exploración y explotación; la condición GLIE garantiza convergencia si visitas todo lo suficiente y vuelves la política codiciosa con el tiempo
- SARSA es on-policy: aprende el valor de la política que realmente sigue, incluida su exploración, lo que la hace más prudente
- Q-learning es off-policy: aprende el valor de la política óptima aunque explore de otra forma, lo que la hace más arriesgada durante el entrenamiento pero óptima al final
- El ejemplo del acantilado lo muestra con claridad: SARSA acumula más recompensa mientras entrena, Q-learning encuentra la ruta más corta
- El operador máximo de Q-learning introduce sesgo de sobreestimación; Double Q-learning lo corrige separando selección y evaluación en dos tablas


## Cierre y próximos pasos

Hoy convertiste la evaluación de la clase anterior en control: dos algoritmos, la misma pregunta ("¿qué tan buena es esta acción?"), dos respuestas distintas según si aprendes el valor de lo que haces (SARSA) o el valor de lo óptimo (Q-learning).

Hasta ahora guardaste todos los valores Q en una tabla, una casilla por cada combinación de estado y acción. Esa tabla funciona perfectamente en una cuadrícula de 12 columnas, pero deja de ser viable en cuanto el número de estados se dispara —por ejemplo, cuando el estado son los píxeles de una pantalla de videojuego—. En la Clase 7 vas a sustituir esa tabla por una red neuronal que aprende a aproximar Q(s,a): así nacen las redes Q profundas (DQN, de *Deep Q-Networks*).

**Antes de la próxima clase:**
- **Abre** el notebook de práctica `clase06_practica_control_sin_modelo.ipynb` y completa los dos ejercicios
- **Experimenta** subiendo `epsilon` a 0.3 en las dos funciones de entrenamiento: observa si la brecha entre la recompensa de SARSA y la de Q-learning se agranda o se reduce
- **Cambia** la penalización del acantilado en `paso` (por ejemplo, de −100 a −10 o a −500) y compara cómo cambian las dos rutas finales

> 🎙️ **Guion de clase:** cierra repitiendo la frase clave: "recompensa inmediata + valor descontado del estado siguiente, para la acción que sigo (SARSA) o para la mejor acción posible (Q-learning)". Es la misma estructura de TD que ya vieron en la clase anterior, ahora aplicada a mejorar la política en vez de solo medirla.
